# 第7章：建立聊天應用程式
## Microsoft Foundry Models API 快速入門

此筆記本改編自包含存取 [Azure OpenAI](notebook-azure-openai.ipynb) 服務的筆記本的 [Azure OpenAI 範例庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)。

> **注意：** GitHub Models 將於2026年7月底退役。此筆記本現目標為 [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst)，提供相同免費試用的模型目錄及 Azure AI 推論 SDK 體驗。


# 概覽  
「大型語言模型是將文本映射到文本的函數。給定一個輸入的文本字串，大型語言模型會嘗試預測接下來的文字」（1）。此「快速入門」筆記本將為使用者介紹大型語言模型的高階概念、開始使用 AML 所需的核心套件需求、提示設計的簡單介紹，以及幾個不同使用案例的短範例。 


## 目錄  

[概覽](#overview)  
[如何使用 OpenAI 服務](#how-to-use-openai-service)  
[1. 建立你的 OpenAI 服務](#1.-creating-your-openai-service)  
[2. 安裝](#2.-installation)    
[3. 認證](#3.-credentials)  

[使用案例](#use-cases)    
[1. 摘要文本](#1.-summarize-text)  
[2. 分類文本](#2.-classify-text)  
[3. 生成新產品名稱](#3.-generate-new-product-names)  
[4. 微調分類器](#4.fine-tune-a-classifier)  

[參考資料](#references)


### 建立你的第一個提示  
這個簡短的練習將提供一個基礎介紹，說明如何在 Microsoft Foundry Models 中提交提示給模型，針對簡單的任務「摘要」進行操作。  


<strong>步驟</strong>：  
1. 如果尚未安裝，請在你的 Python 環境中安裝 `azure-ai-inference` 函式庫。  
2. 載入標準輔助函式庫並設定你的 Microsoft Foundry Models 憑證。  
3. 為你的任務選擇一個模型  
4. 為模型建立一個簡單的提示  
5. 將你的請求提交給模型 API！  


### 1. 安裝 `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. 匯入輔助程式庫並實例化憑證


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. 尋找合適的模型  
像 GPT-4o 和 GPT-4o mini 這類模型能理解和生成自然語言，並且可在 Microsoft Foundry Models 目錄中找到，同時還有來自 Meta、Mistral、Cohere 及 Microsoft 的模型。


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. 提示設計  

「大型語言模型的魔力在於，透過在大量文本上進行預測誤差最小化的訓練，模型最終學會了對這些預測有用的概念。例如，它們學會了像這樣的概念」(1):

* 如何拼寫
* 語法如何運作
* 如何改寫
* 如何回答問題
* 如何進行對話
* 如何用多種語言寫作
* 如何編程
* 等等。

#### 如何控制大型語言模型  
「在所有輸入到大型語言模型的資料中，最具影響力的絕對是文字提示」(1)。

大型語言模型可以通過幾種方式被提示以產生輸出：

指示：告訴模型你想要什麼
補全：誘導模型補全你想要的開頭
示範：向模型展示你想要的內容，透過：
提示中的幾個範例
微調訓練資料集中數百或數千個範例」



#### 創建提示的三個基本準則：

<strong>展示並告訴</strong>。清楚表達你想要什麼，透過指示、範例或兩者結合。如果你想讓模型對一串項目按字母順序排序，或根據情感分類一段文字，請展示給它知道這就是你想要的。

<strong>提供優質資料</strong>。如果你嘗試建構分類器或讓模型遵循某個模式，確保有足夠的範例。務必校對你的範例——模型通常足夠聰明可以識別基本拼寫錯誤並給你回應，但它也可能假設這是故意的，這會影響回應結果。

**檢查你的設定。** 溫度 (temperature) 和 top_p 設定控制模型產生回應的確定性。如果你要求的回應只有一個正確答案，則應將這些設定調低；如果你希望獲得更多樣化的回應，則可以將它們調高。人們對這些設定最大的誤解是認為它們是「聰明度」或「創造力」的控制手段。


資料來源：https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. 遞交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### 重複相同的呼叫，結果會如何比較？ 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## 總結文本  
#### 挑戰  
透過在文本段落末尾加上「tl;dr:」來總結文本。注意模型如何能在沒有額外指令的情況下理解並執行多種任務。你可以嘗試使用比tl;dr更具描述性的提示來改變模型行為及定制你所獲得的摘要(3)。  

近期研究顯示，先在大型語料庫上進行預訓練，然後在特定任務上微調，可在多項自然語言處理任務和基準上取得顯著進展。雖然此方法通常架構上與任務無關，但仍需數千或數萬個特定任務的微調數據集。相比之下，人類通常只需少量示例或簡單指示就能完成新語言任務——這正是當前自然語言處理系統大多仍難以做到的。本文展示了擴大語言模型規模能大幅提升任務無關的少量示例學習表現，有時甚至達到與先前最先進微調方法相當的水平。 



摘要  


# 適用於多種用例的練習  
1. 摘要文本  
2. 文本分類  
3. 產生新產品名稱


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 分類文本  
#### 挑戰  
將項目分類至推理時提供的類別。在以下範例中，我們在提示中同時提供了類別與要分類的文本(*playground_reference)。 

客戶查詢：您好，我的筆記本電腦鍵盤上的一個鍵最近壞了，我需要更換： 

分類類別：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 生成新產品名稱
#### 挑戰
根據範例詞語創建產品名稱。這裡我們在提示中包含了有關我們將生成名稱的產品的信息。 我們還提供了類似的範例來展示我們希望收到的模式。 我們還將溫度值設得較高，以提高隨機性並獲得更具創新性的回應。

產品描述：家用奶昔製造機
種子詞：快速、健康、緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：一雙適合任何腳型的鞋子。
種子詞：適應性、合腳、全合腳。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# 參考資料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [調整 GPT 模型以分類文本的最佳做法](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 如需更多協助  
[OpenAI 商業化團隊](AzureOpenAITeam@microsoft.com) 


# 貢獻者
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
